In [ ]:
import numpy as np
import pandas as pd
import os
from IPython.display import display
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import tensorflow as tf
import gpflow
from GPcounts.RNA_seq_GP import rna_seq_gp

In [ ]:
counts_df = pd.read_csv(r"all_normalised_counts.csv")
sampletable_df = pd.read_csv(r"all_sampletable.csv")

In [ ]:
#CSV processing 

counts_df = counts_df.rename(columns={"Unnamed: 0": "ID"})
counts_df.set_index(counts_df.columns[0], inplace=True)

#Reduces the impact of cholesky decomposition failure, a common error for GPcounts 

counts_df = counts_df.round().astype(int)

# Filter out genes with 15 or fewer observations identical to masigpro (nonzero counts)

counts_df = counts_df.loc[(df.iloc[:, 1:] > 0).sum(axis=1) > 15]

In [ ]:
X = sampletable_df[["Time"]]
Y = counts_df.index.tolist()

In [ ]:
# Two-sample test 
# 10 inducing points and sparse inference to improve computational efficiency [8-44]. 
# safe_mode = false

sparse = True
gp_counts = rna_seq_gp(X,Y.loc[gene_names], M=10, sparse = sparse, safe_mode=False) 
lik_name = "Negative_binomial" 
results = gp_counts.Two_samples_test(lik_name)
results_df = pd.DataFrame(results)

In [ ]:
#Statistical Tests

llr_values = results.loc[:,"log_likelihood_ratio"]

degrees_freedom = 1
p_values = [1 - stats.chi2.cdf(abs(llr), degrees_freedom) for llr in llr_values]

results_df_chisq = 1
results_df["p_value"] = 1 - stats.chi2.cdf(abs(results_df["log_likelihood_ratio"]), results_df_chisq)

p_value = results_df["p_value"].values
rejected, q_values, _, _ = multipletests(p_values, method='fdr_bh')
results_df["q_value"] = q_values
results_df["significant"] = rejected

results_df.to_csv("all_gpcounts_significant_genes.csv", index=True)

In [ ]:
#Kernel fitting

counts_array = Y.values
counts_array = np.array(Y.values, dtype=np.float64)

time_points_array = np.array(X.values, dtype=np.float64)
time_points_array = time_points_array.reshape(-1,1)

rbf_kernel = gpflow.kernels.SquaredExponential()
periodic_kernel = gpflow.kernels.Periodic(base_kernel=rbf_kernel)
combined_kernel = rbf_kernel + periodic_kernel

#Included due to GPcounts' tendency to fail 

file_path = "Only_RBF_Periodicity_Values.csv"
if os.path.exists(file_path):
    os.remove(file_path)

In [ ]:
results = []

for gene_index in range(counts_array.shape[0]):
    
    y = counts_array[gene_index, :].reshape(-1, 1)

    y = tf.convert_to_tensor(y, dtype=tf.float64)

    rbf_value = rbf_kernel(time_points_array, time_points_array)
    periodic_value = periodic_kernel(time_points_array, time_points_array)
    combined_value = rbf_value + periodic_value

    rbf_kernel= gpflow.kernels.SquaredExponential()
    periodic_kernel = gpflow.kernels.Periodic(base_kernel=gpflow.kernels.SquaredExponential())
    combined_kernel = rbf_kernel + periodic_kernel 

    model = gpflow.models.GPR(data=(time_points_array, y), kernel=combined_kernel)
    gpflow.optimizers.Scipy().minimize(model.training_loss, model.trainable_variables)
    
    ratio = rbf_value/combined_value

    results.append({
        "Gene Name" : gene_index,
        "RBF Value": np.mean(rbf_value.numpy()),
        "Peridodicity Value": np.mean(periodic_value.numpy()),
        "Combined Value": np.mean(combined_value.numpy()),
        "Ratio RBF:Periodicity": np.mean(ratio.numpy())
    })

In [ ]:
kernel_df = pd.DataFrame(results)
kernel_df.to_csv("all_rbf_periodicity_values.csv", index=False, mode="w")
existing_df = pd.read_csv("all_gpcounts_significant_genes.csv")

updated_df = pd.concat([existing_df, kernel_df.iloc[:, 1:]], axis=1)
updated_df = updated_df.round(3)
updated_df.to_csv("final_gpcounts.csv", index=False)

In [ ]:
#Visualisation of results

from helper import plot 

test_name = 'Two_samples_test'
params = gp_counts.load_predict_models(gene_names,test_name,lik_name)
plot(params,X.values,Y.loc[gene_names],results, sparse = sparse)